Import libraries

In [195]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns

Load the datasets

In [196]:
students = pd.read_csv('C:\\Users\\vikes\\Favorites\\Links\\100daysinC\\explo_project\\Students_Raw.csv')
courses = pd.read_csv('C:\\Users\\vikes\\Favorites\\Links\\100daysinC\\explo_project\\Course_Catalogue.csv')
interactions = pd.read_csv('C:\\Users\\vikes\\Favorites\\Links\\100daysinC\\explo_project\\Student_Course_Interactions.csv')
dictionary = pd.read_csv('C:\\Users\\vikes\\Favorites\\Links\\100daysinC\\explo_project\\Data_Dictionary.csv')

Create working copies

In [197]:
students_raw = students.copy()
courses_raw = courses.copy()
interactions_raw = interactions.copy()
dictionary_raw = dictionary.copy()

Check initial Data dimensions

In [198]:
datasets = {
    "Students": students,
    "Courses": courses,
    "Interactions": interactions
}

for name, df in datasets.items():
    print(f"\n{name}")
    print("Shape:", df.shape)
    print("Duplicate rows:", df.duplicated().sum())
    print("Missing values:", df.isnull().sum().sum())


Students
Shape: (1020, 33)
Duplicate rows: 40
Missing values: 1455

Courses
Shape: (178, 20)
Duplicate rows: 0
Missing values: 288

Interactions
Shape: (5869, 10)
Duplicate rows: 114
Missing values: 5104


Handle Missing value placeholders

In [199]:
missing_tokens = ["NA", "N/A", "null", "nan", "?", "Not Available", " ","na", "n/a", "Null", "NaN", "not available", "none", "None"]
def standardize_missing(df):
    df = df.copy()
    for col in df.select_dtypes(include=["object", "string"]).columns:
        df[col] = df[col].astype("string").str.strip()

        df[col] = df[col].mask(
            df[col].str.casefold().isin(missing_tokens), pd.NA
        )
    return df

students = standardize_missing(students_raw)
courses = standardize_missing(courses_raw)
interactions = standardize_missing(interactions_raw)
dictionary = standardize_missing(dictionary_raw)

print("Missing value placeholders standardized.")

Missing value placeholders standardized.


Remove exact duplicate rows

In [200]:
print("Before removing duplicates:")

print("Students", students.duplicated().sum())
print("Courses", courses.duplicated().sum())
print("Interactions", interactions.duplicated().sum())

students = students.drop_duplicates()
courses = courses.drop_duplicates()
interactions = interactions.drop_duplicates()

print("After removing duplicates:")
print("Students", students.duplicated().sum())
print("Courses", courses.duplicated().sum())
print("Interactions", interactions.duplicated().sum())

Before removing duplicates:
Students 40
Courses 0
Interactions 114
After removing duplicates:
Students 0
Courses 0
Interactions 0


Standardize text formatting

In [201]:
def clean_text(df):
    df = df.copy()

    for col in df.select_dtypes(include="object").columns:
        df[col] = (
            df[col].astype("string").str.strip().str.replace(r"\s+", " ", regex= True)
        )
    return df

students = clean_text(students)
courses = clean_text(courses)
interactions = clean_text(interactions)

print("Text formatting standardized.")

Text formatting standardized.


Standardize important identifiers

In [202]:
#Student IDs
students["Student_ID"] = students["Student_ID"].str.upper().str.strip()
interactions["Student_ID"] = interactions["Student_ID"].str.upper().str.strip()

# Course IDs
courses["Course_ID"] = courses["Course_ID"].str.upper().str.strip()
interactions["Course_ID"] = interactions["Course_ID"].str.upper().str.strip()

#Course codes
courses["Course Code"] = courses["Course Code"].str.upper().str.strip()
interactions["Course_Code"] = interactions["Course_Code"].str.upper().str.strip()
students["Selected_Course_Code"] = students["Selected_Course_Code"].str.upper().str.strip()

print("Identifier formatting completed.")

Identifier formatting completed.


Convert numerical columns

In [203]:
student_numeric_cols = [
    "Age",
    "CGPA",
    "Previous_Semester_SGPA",
    "Attendance_Percentage",
    "Weekly_Study_Hours",
    "Num_Prior_Exploratory_Courses",
    "Num_Certifications",
    "Confusion_Level",
    "Selection_Confidence",
    "Previous_Course_Satisfaction"
]

for col in student_numeric_cols:
    students[col] = pd.to_numeric(
        students[col], errors="coerce"
    )

interaction_numeric_cols = [
    "Time_Spent_Minutes",
    "Rating_Given"
]

for col in interaction_numeric_cols:
    interactions[col] = pd.to_numeric(
        interactions[col], errors="coerce"
    )
course_numeric_cols = [
    "AY", "CREDIT", "Difficulty_Level", "Popularity_Index"
]

for col in course_numeric_cols:
    courses[col] = pd.to_numeric(
        courses[col], errors="coerce"
    )

print("Numerical data types converted.")

# Validate interaction time
invalid_time = interactions["Time_Spent_Minutes"] < 0
print("Invalid interaction time values:", invalid_time.sum())
interactions.loc[invalid_time, "Time_Spent_Minutes"] = np.nan

Numerical data types converted.
Invalid interaction time values: 20


Handle invalid numerical values

In [204]:

student_ranges = {
    "Age": (17, 30),
    "CGPA": (0, 10),
    "Previous_Semester_SGPA": (0, 10),
    "Attendance_Percentage": (0, 100),
    "Weekly_Study_Hours": (0, 60),
    "Num_Prior_Exploratory_Courses": (0, 6),
    "Num_Certifications": (0, 8),
    "Confusion_Level": (1, 10),
    "Selection_Confidence": (1, 10),
    "Previous_Course_Satisfaction": (1, 5)
}

for col, (low, high) in student_ranges.items():
    invalid = (
        (students[col] < low) |
        (students[col] > high)
    )
    print(col, "invalid values:", invalid.sum())
    students.loc[invalid, col] = np.nan

#validate interaction ratings

invalid_rating = (
    (interactions["Rating_Given"] < 1) |
    (interactions["Rating_Given"] > 5)
)
print("Invalid ratings:", invalid_rating.sum())
interactions.loc[invalid_rating, "Rating_Given"] = np.nan

#validate course catalogue fields

for col in ["Difficulty_Level", "Popularity_Index"]:
    invalid = (courses[col] < 1) | (courses[col] > 5)
    print(col, "invalid values:", invalid.sum())
    courses.loc[invalid, col] = np.nan
invalid_credit = courses["CREDIT"] <= 0
print("Invalid credits:", invalid_credit.sum())
courses.loc[invalid_credit, "CREDIT"] = np.nan


Age invalid values: 16
CGPA invalid values: 13
Previous_Semester_SGPA invalid values: 4
Attendance_Percentage invalid values: 12
Weekly_Study_Hours invalid values: 14
Num_Prior_Exploratory_Courses invalid values: 10
Num_Certifications invalid values: 12
Confusion_Level invalid values: 17
Selection_Confidence invalid values: 13
Previous_Course_Satisfaction invalid values: 13
Invalid ratings: 11
Difficulty_Level invalid values: 0
Popularity_Index invalid values: 0
Invalid credits: 0


Standardize categorical labels

In [205]:
student_category_cols = [
    "Gender",
    "Class12_Stream",
    "Skill_Level",
    "Learning_Preference",
    "Preferred_Difficulty",
    "Workload_Preference",
    "Preferred_Slot",
    "Internship_Experience"
]

for col in student_category_cols:
    students[col] = students[col].str.strip().str.lower()

interaction_category_cols = [
    "Interaction_Type",
    "Recommended_By",
    "Suitability_Label"
]

for col in interaction_category_cols:
    interactions[col] = interactions[col].str.strip().str.lower()

print("Selected categorical labels standardized.")


Selected categorical labels standardized.


Convert data columns

In [206]:

students["Survey_Timestamp"] = pd.to_datetime(
    students["Survey_Timestamp"],
    format="mixed",
    errors="coerce"
)

interactions["Interaction_Date"] = pd.to_datetime(
    interactions["Interaction_Date"],
    format="mixed",
    errors="coerce"
)

print("Date conversion completed.")

print("Invalid student dates:",
      students["Survey_Timestamp"].isna().sum())

print("Invalid interaction dates:",
      interactions["Interaction_Date"].isna().sum())


Date conversion completed.
Invalid student dates: 62
Invalid interaction dates: 59


Handle missing numerical values

In [207]:
for col in student_numeric_cols:
    median_value = students[col].median()
    if pd.notna(median_value):
        students[col] = students[col].fillna(median_value)

print("Student numerical missing values handled.")

#interaction time:

median_time = interactions["Time_Spent_Minutes"].median()
if pd.notna(median_time):
    interactions["Time_Spent_Minutes"] = (
        interactions["Time_Spent_Minutes"].fillna(median_time)
    )

print("Interaction time missing values handled.")


Student numerical missing values handled.
Interaction time missing values handled.


Handle missing categorical values

In [208]:
categorical_cols = students.select_dtypes(
    include=["object", "string"]
).columns

for col in categorical_cols:
    if students[col].isnull().sum() > 0:
        mode_value = students[col].mode(dropna=True)[0]
        students[col] = students[col].fillna(mode_value)

# Handle optional course information

courses["Remarks"] = courses["Remarks"].fillna("Not Specified")
courses["Technical_Prerequisite"] = (
    courses["Technical_Prerequisite"].fillna("Not Specified")
)

# Handle missing optional information

interactions["Recommended_By"] = (
    interactions["Recommended_By"].fillna("Not Specified")
)

Inspect duplicate identifiers

In [209]:
print("Duplicate Student IDs:",
      students["Student_ID"].duplicated().sum())

print("Duplicate Interaction IDs:",
      interactions["Interaction_ID"].duplicated().sum())

print("Duplicate Course IDs:",
      courses["Course_ID"].duplicated().sum())

student_duplicate_records = students[
    students["Student_ID"].duplicated(keep=False)
].sort_values(by="Student_ID")

interaction_duplicate_records = interactions[
    interactions["Interaction_ID"].duplicated(keep=False)
].sort_values(by="Interaction_ID")

course_duplicate_records = courses[
    courses["Course_ID"].duplicated(keep=False)
].sort_values(by="Course_ID")

print("Duplicate Student IDs:", students['Student_ID'].duplicated().sum())
print("Duplicate Interaction IDs:", interactions['Interaction_ID'].duplicated().sum())
print("Duplicate Course IDs:", courses['Course_ID'].duplicated().sum())

display(student_duplicate_records.head(10))
display(interaction_duplicate_records.head(10))
display(course_duplicate_records.head(10))


Duplicate Student IDs: 20
Duplicate Interaction IDs: 22
Duplicate Course IDs: 0
Duplicate Student IDs: 20
Duplicate Interaction IDs: 22
Duplicate Course IDs: 0


,Student_ID,Batch_Year,Semester,Branch,School,Campus,Age,Gender,Class12_Stream,CGPA,...,Preferred_Difficulty,Workload_Preference,Preferred_Slot,Confusion_Level,Confusion_Reason,Selection_Confidence,Previous_Course_Satisfaction,Survey_Timestamp,Selected_Course_Code,Selected_Exploratory_Course
30,STU0028,2024,5,B.Tech CSE - Data Science,SOCS,bidholi,20.0,male,pcm,6.96,...,easy,low,slot 2,7.0,Unclear course content,3.0,3.0,NaT,DIGB2003P,Introduction to Cloud Computing for Managers
235,STU0028,2024,5,B.Tech CSE - Data Science,SOCS,bidholi,20.0,male,pcm,6.96,...,easy,low,slot 2,8.0,Unclear course content,3.0,3.0,NaT,DIGB2003P,Introduction to Cloud Computing for Managers
32,STU0030,2025,3,B.Tech CSE - Data Science,SOAE,Kandoli,20.0,male,pcmb,8.02,...,challenging,low,no preference,5.0,Unclear course content,7.0,3.0,2026-09-22 10:02:06,PEGS2057,Remote Sensing and GIS
343,STU0030,2025,3,B.TECH CSE - DATA SCIENCE,SOAE,Kandoli,20.0,male,pcmb,8.02,...,challenging,low,no preference,6.0,Unclear course content,7.0,3.0,2026-09-23 21:02:06,PEGS2057,Remote Sensing and GIS
934,STU0113,2023,7,B.TECH CSE - CORE,School of Computer Science,Bidholi Campus,21.0,male,pcm,6.97,...,moderate,high,slot 2,3.0,Unclear course content,8.0,3.0,2026-08-27 20:38:02,SMBA1018,Creative Writing
122,STU0113,2023,7,B.Tech CSE - Core,School of Computer Science,Bidholi Campus,21.0,male,pcm,6.97,...,moderate,high,slot 2,4.0,Unclear course content,8.0,3.0,2026-08-27 20:38:02,SMBA1018,Creative Writing
442,STU0170,2025,3,b tech cse - data science,SOCS,Kandoli,19.0,male,pcm,6.76,...,mod,med,no preference,3.0,Too many choices,8.0,2.0,2026-08-28 06:55:15,FINC2066,Money and Banking
187,STU0170,2025,3,B Tech CSE - Data Science,SOCS,Kandoli,19.0,male,pcm,6.76,...,mod,med,no preference,4.0,Too many choices,8.0,2.0,2026-08-25 22:55:15,FINC2066,Money and Banking
717,STU0178,2023,6,b.tech electrical engg,SOAE,Bidholi,20.0,female,pcm,6.76,...,challenging,medium,no preference,4.0,Interests not matching available courses,7.0,2.0,2026-08-14 00:00:00,DIGB3002P,Mobile & email marketing
203,STU0178,2023,6,B.Tech Electrical Engg,SOAE,Bidholi,20.0,female,pcm,6.76,...,challenging,medium,no preference,5.0,Interests not matching available courses,7.0,2.0,2026-08-14 00:00:00,DIGB3002P,Mobile & email marketing


,Interaction_ID,Student_ID,Course_ID,Course_Code,Interaction_Type,Interaction_Date,Time_Spent_Minutes,Recommended_By,Rating_Given,Suitability_Label
13,INT00014,STU0003,C047,ECEG2059,compared,2026-07-25 00:00:00,38,faculty/advisor,<NA>,low
2127,INT00014,STU0003,C138,DSBA2009,compared,2026-07-25 00:00:00,38,faculty/advisor,<NA>,low
212,INT00207,STU0036,C064,SDCS2007,viewed,2026-08-13 07:55:00,7,online (youtube/reddit),<NA>,<NA>
3884,INT00207,STU0036,C013,SOB20B001,viewed,2026-08-13 07:55:00,7,online (youtube/reddit),<NA>,<NA>
3292,INT00985,STU0166,C108,SUEN3019,enrolled,2026-07-20 00:39:58,3,,<NA>,low
1009,INT00985,STU0166,C173,MKTG3011,enrolled,2026-07-20 00:39:58,3,,<NA>,low
1098,INT01072,STU0180,C095,PEAU2020,enrolled,2026-10-18 00:00:00,2,friend,2.0,low
1902,INT01072,STU0180,C175,INTB3019,enrolled,2026-10-18 00:00:00,2,friend,2.0,low
5577,INT01118,STU0188,C148,SMJM3005,viewed,2026-07-29 00:54:14,3,friend,<NA>,medium
1144,INT01118,STU0188,C019,MKTG2035,viewed,2026-07-29 00:54:14,3,friend,<NA>,medium


,Course_ID,AY,School,Sem,Course Code,Course,CREDIT,Minor Stream,Slot,Schedule,Remarks,Primary_Domain,Secondary_Domain,Difficulty_Level,Workload_Level,Teaching_Style,Technical_Prerequisite,Popularity_Index,Skill_Tags,Typical_Career_Paths


saving the records

In [210]:

import os
os.makedirs("../Cleaned_Data", exist_ok=True)

student_duplicate_records.to_csv(
    "../Cleaned_Data/Student_Duplicate_ID_Review.csv",
    index=False
)

interaction_duplicate_records.to_csv(
    "../Cleaned_Data/Interaction_Duplicate_ID_Review.csv",
    index=False
)

course_duplicate_records.to_csv(
    "../Cleaned_Data/Course_Duplicate_ID_Review.csv",
    index=False
)

Check missing values after cleaning

In [211]:

cleaned_datasets = {
    "Students": students,
    "Courses": courses,
    "Interactions": interactions
}

for name, df in cleaned_datasets.items():

    print(f"\n========== {name} ==========")

    print("Remaining missing values:",
          df.isnull().sum().sum())

    print("\nMissing values column-wise:")

    print(
        df.isnull().sum()[
            df.isnull().sum() > 0
        ]
    )


========== Students ==========
Remaining missing values: 62

Missing values column-wise:
Survey_Timestamp    62
dtype: int64

========== Courses ==========
Remaining missing values: 0

Missing values column-wise:
Series([], dtype: int64)

========== Interactions ==========
Remaining missing values: 4940

Missing values column-wise:
Interaction_Date       59
Rating_Given         4628
Suitability_Label     253
dtype: int64


Validate relationships between datasets

In [212]:
# Identify unmatched Student IDs
student_ids = set(students["Student_ID"].dropna())

unmatched_student_records = interactions[
    ~interactions["Student_ID"].isin(student_ids)
]

# Identify unmatched Course IDs
course_ids = set(courses["Course_ID"].dropna())

unmatched_course_records = interactions[
    ~interactions["Course_ID"].isin(course_ids)
]

# Identify unmatched Course Codes
catalogue_codes = set(courses["Course Code"].dropna())

unmatched_selected_codes = students[
    students["Selected_Course_Code"].notna() &
    ~students["Selected_Course_Code"].isin(catalogue_codes)
]

unmatched_interaction_codes = interactions[
    interactions["Course_Code"].notna() &
    ~interactions["Course_Code"].isin(catalogue_codes)
]

print("Unmatched Student ID records:",
      len(unmatched_student_records))

print("Unmatched Course ID records:",
      len(unmatched_course_records))

print("Unmatched selected course codes:",
      len(unmatched_selected_codes))

print("Unmatched interaction course codes:",
      len(unmatched_interaction_codes))

# Export records for manual review
unmatched_student_records.to_csv(
    "../Cleaned_Data/Unmatched_Student_Records.csv", index=False
)

unmatched_course_records.to_csv(
    "../Cleaned_Data/Unmatched_Course_Records.csv", index=False
)

unmatched_selected_codes.to_csv(
    "../Cleaned_Data/Unmatched_Selected_Codes.csv", index=False
)

unmatched_interaction_codes.to_csv(
    "../Cleaned_Data/Unmatched_Interaction_Codes.csv", index=False
)

Unmatched Student ID records: 214
Unmatched Course ID records: 0
Unmatched selected course codes: 17
Unmatched interaction course codes: 88


Compare raw and cleaned datasets

In [213]:

comparison = []
raw_datasets = {
    "Students": students_raw,
    "Courses": courses_raw,
    "Interactions": interactions_raw
}
for name in raw_datasets:
    raw_df = raw_datasets[name]
    clean_df = cleaned_datasets[name]
    comparison.append({
        "Dataset": name,
        "Raw Rows": len(raw_df),
        "Cleaned Rows": len(clean_df),
        "Raw Columns": len(raw_df.columns),
        "Cleaned Columns": len(clean_df.columns),
        "Raw Missing Values": raw_df.isnull().sum().sum(),
        "Cleaned Missing Values": clean_df.isnull().sum().sum(),
        "Raw Duplicates": raw_df.duplicated().sum(),
        "Cleaned Duplicates": clean_df.duplicated().sum()
    })
comparison_df = pd.DataFrame(comparison)
display(comparison_df)

,Dataset,Raw Rows,Cleaned Rows,Raw Columns,Cleaned Columns,Raw Missing Values,Cleaned Missing Values,Raw Duplicates,Cleaned Duplicates
0,Students,1020,980,33,33,1455,62,40,0
1,Courses,178,178,20,20,288,0,0,0
2,Interactions,5869,5755,10,10,5104,4940,114,0


Export cleaned datasets

In [214]:
import os
os.makedirs("../Cleaned_Data", exist_ok=True)
students.to_csv(
    "../Cleaned_Data/Students_Cleaned.csv",
    index=False
)
courses.to_csv(
    "../Cleaned_Data/Course_Catalogue_Cleaned.csv",
    index=False
)
interactions.to_csv(
    "../Cleaned_Data/Student_Course_Interactions_Cleaned.csv",
    index=False
)
comparison_df.to_csv(
    "../Cleaned_Data/Cleaning_Comparison.csv",
    index=False
)
print("All cleaned datasets saved successfully!")

All cleaned datasets saved successfully!


Final Verification

In [215]:
print("FINAL DATASET DIMENSIONS")
for name, df in cleaned_datasets.items():
    print(f"{name}: {df.shape}")
print("\nExact duplicate rows after cleaning")
for name, df in cleaned_datasets.items():
    print(name, ":", df.duplicated().sum())

FINAL DATASET DIMENSIONS
Students: (980, 33)
Courses: (178, 20)
Interactions: (5755, 10)

Exact duplicate rows after cleaning
Students : 0
Courses : 0
Interactions : 0


In [216]:
print("Student Dataset Summary")
display(students.describe(include="all").T)

print("Course dataset summary")
display(courses.describe(include="all").T)

print("Interaction dataset summary")
display(interactions.describe(include="all").T)

Student Dataset Summary


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
Student_ID,980,960,STU0234,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Batch_Year,980,11,2025,443,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Semester,980,23,3,381,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Branch,980,148,B.Tech CSE - Data Science,108,NaN,NaN,NaN,NaN,NaN,NaN,NaN
School,980,32,SOCS,261,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Campus,980,13,Bidholi,460,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Age,980.0,<NA>,<NA>,<NA>,20.264286,17.0,20.0,20.0,21.0,27.0,1.009935
Gender,980,10,male,604,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Class12_Stream,980,17,pcm,521,NaN,NaN,NaN,NaN,NaN,NaN,NaN
CGPA,980.0,<NA>,<NA>,<NA>,7.533071,0.0,6.9775,7.59,8.14,10.0,1.071788


Course dataset summary


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Course_ID,178,178,C001,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AY,178.0,NaN,NaN,NaN,2024.224719,0.724905,2023.0,2024.0,2024.0,2025.0,2025.0
School,178,7,SOAE,71,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Sem,178,10,3,63,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Course Code,178,156,ECEG2059,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Course,178,155,Industrial Power Electronics,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
CREDIT,178.0,NaN,NaN,NaN,3.134831,0.491533,2.0,3.0,3.0,3.0,6.0
Minor Stream,178,47,Minor in Electronics Engineering,11,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Slot,178,3,Slot 1,90,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Schedule,178,5,Monday to Wednesday- 11:00 AM to 12:00 Noon,55,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Interaction dataset summary


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
Interaction_ID,5755,5733,INT00207,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Student_ID,5755,1074,STU0516,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Course_ID,5755,166,C037,169,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Course_Code,5755,200,CSIB1002,166,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Interaction_Type,5755,13,viewed,2272,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Interaction_Date,5696,NaN,NaN,NaN,2026-08-26 09:37:55.114290688,2026-01-01 10:00:00,2026-08-03 15:29:46.500000,2026-08-28 23:45:30,2026-09-24 00:00:00,2027-01-10 00:00:00,NaN
Time_Spent_Minutes,5755.0,<NA>,<NA>,<NA>,42.148045,0.0,3.0,5.0,8.0,9999.0,547.764062
Recommended_By,5755,20,self-search,1669,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Rating_Given,1127.0,<NA>,<NA>,<NA>,4.038154,1.0,3.0,4.0,5.0,5.0,1.001491
Suitability_Label,5502,10,high,2121,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Final Quality check

In [217]:
print("FINAL DATA QUALITY CHECK")

for name, df in cleaned_datasets.items():

    print(f"\n{name}")
    print("Shape:", df.shape)
    print("Exact duplicate rows:", df.duplicated().sum())
    print("Total missing values:", df.isnull().sum().sum())

    print("\nMissing values by column:")
    print(df.isnull().sum()[df.isnull().sum() > 0])

# Check invalid interaction time
print("\nNegative interaction times:",
      (interactions["Time_Spent_Minutes"] < 0).sum())

# Check invalid ratings
print("Invalid ratings:",
      ((interactions["Rating_Given"] < 1) |
       (interactions["Rating_Given"] > 5)).sum())

FINAL DATA QUALITY CHECK

Students
Shape: (980, 33)
Exact duplicate rows: 0
Total missing values: 62

Missing values by column:
Survey_Timestamp    62
dtype: int64

Courses
Shape: (178, 20)
Exact duplicate rows: 0
Total missing values: 0

Missing values by column:
Series([], dtype: int64)

Interactions
Shape: (5755, 10)
Exact duplicate rows: 0
Total missing values: 4940

Missing values by column:
Interaction_Date       59
Rating_Given         4628
Suitability_Label     253
dtype: int64

Negative interaction times: 0
Invalid ratings: 0
